# Phase 2 - Unsupervised Learning

## Goal

The goal of this notebook is to use clustering to group users into similar profiles.

We use K-Means clustering because it is simple and useful for grouping users based on numerical and encoded categorical features. The target label is removed before clustering.

After clustering, we check the clusters using:

- WCSS
- Silhouette Score
- BCubed Precision and Recall
- PCA visualization
- Cluster profile summary


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")


In [ ]:
# Load dataset
df = pd.read_csv("Dataset/insurance.csv")
df = df.drop_duplicates().reset_index(drop=True)
display(df.head())

In [ ]:

# Create risk_level only for later checking.
# It will not be used while training the clustering model.

risk_names = ["Low Cost Risk", "Medium Cost Risk", "High Cost Risk"]

df["risk_level"] = pd.qcut(
    df["charges"],
    q=3,
    labels=risk_names
)

print(df["risk_level"].value_counts().reindex(risk_names))


## Preparing Data for Clustering

For clustering, we remove the class label and target column. We only use user information:

- age
- sex
- bmi
- children
- smoker
- region

This follows the requirement that the class label should be removed before clustering.


In [ ]:

# Prepare features for clustering
cluster_features = ["age", "sex", "bmi", "children", "smoker", "region"]
X_cluster_raw = df[cluster_features]

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

preprocess_cluster = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_cluster = preprocess_cluster.fit_transform(X_cluster_raw)

# Convert to dense array for PCA and clustering display
if hasattr(X_cluster, "toarray"):
    X_cluster = X_cluster.toarray()

print("Clustering feature matrix shape:", X_cluster.shape)


## Choosing Number of Clusters

We test different values of K. WCSS helps us see the elbow point. Silhouette Score helps us measure how well separated the clusters are.


In [ ]:

# Test different K values
k_values = range(2, 9)
wcss = []
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(X_cluster)

    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster, cluster_labels))

score_df = pd.DataFrame({
    "K": list(k_values),
    "WCSS": wcss,
    "Silhouette Score": silhouette_scores
})

display(score_df.round(4))

best_k = score_df.sort_values(by="Silhouette Score", ascending=False).iloc[0]["K"]
best_k = int(best_k)
print("Best K based on Silhouette Score:", best_k)


In [ ]:

# Plot WCSS and Silhouette Score
plt.figure(figsize=(6, 4))
plt.plot(score_df["K"], score_df["WCSS"], marker="o")
plt.title("Elbow Method - WCSS")
plt.xlabel("Number of Clusters K")
plt.ylabel("WCSS")
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(score_df["K"], score_df["Silhouette Score"], marker="o")
plt.title("Silhouette Score by K")
plt.xlabel("Number of Clusters K")
plt.ylabel("Silhouette Score")
plt.show()


In [ ]:

# Train final K-Means model

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
clusters = kmeans_final.fit_predict(X_cluster)

df_clustered = df.copy()
df_clustered["cluster"] = clusters

print("Final WCSS:", round(kmeans_final.inertia_, 2))
print("Final Silhouette Score:", round(silhouette_score(X_cluster, clusters), 4))
print("Cluster counts:")
print(df_clustered["cluster"].value_counts().sort_index())


In [ ]:

# BCubed Precision and Recall
# We use risk_level only for evaluation after clustering.
# risk_level was not used when training K-Means.

def bcubed_precision_recall(true_labels, cluster_labels):
    true_labels = np.array(true_labels)
    cluster_labels = np.array(cluster_labels)

    precision_scores = []
    recall_scores = []

    for i in range(len(true_labels)):
        same_cluster = cluster_labels == cluster_labels[i]
        same_class = true_labels == true_labels[i]

        correct = np.sum(same_cluster & same_class)

        precision_i = correct / np.sum(same_cluster)
        recall_i = correct / np.sum(same_class)

        precision_scores.append(precision_i)
        recall_scores.append(recall_i)

    return np.mean(precision_scores), np.mean(recall_scores)


bcubed_precision, bcubed_recall = bcubed_precision_recall(
    df_clustered["risk_level"],
    df_clustered["cluster"]
)

print("BCubed Precision:", round(bcubed_precision, 4))
print("BCubed Recall:", round(bcubed_recall, 4))


In [ ]:

# Cluster profile summary
# This helps us understand what each cluster means.

cluster_summary = df_clustered.groupby("cluster").agg(
    count=("cluster", "size"),
    avg_age=("age", "mean"),
    avg_bmi=("bmi", "mean"),
    avg_children=("children", "mean"),
    smoker_rate=("smoker", lambda x: (x == "yes").mean()),
    avg_charges=("charges", "mean")
)

print("Cluster profile summary:")
display(cluster_summary.round(3))

risk_by_cluster = pd.crosstab(
    df_clustered["cluster"],
    df_clustered["risk_level"],
    normalize="index"
) * 100

print("Risk level percentage inside each cluster:")
display(risk_by_cluster.round(2))


In [ ]:

# PCA visualization of clusters
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, s=30)
plt.title("K-Means Clusters using PCA")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(scatter, label="Cluster")
plt.show()

print("Explained variance by PCA components:", np.round(pca.explained_variance_ratio_, 3))


## Unsupervised Results Interpretation

K-Means clustering was used to group users into similar profiles based on age, sex, BMI, children, smoking status, and region.

The class label `risk_level` was removed before training the clustering model. This is important because clustering should find groups without using the target label.

We tested different values of K using WCSS and Silhouette Score. The best K is selected based on the highest Silhouette Score.

After clustering, we compared the clusters with `risk_level` only for evaluation and interpretation.

BCubed Precision and Recall were used to check how well the clusters matched the risk-level groups.

The cluster profile summary helps explain the meaning of each cluster. For example, a cluster with a higher smoker rate or higher average BMI may represent users with higher insurance cost risk.

These clusters can improve the advice system by creating user profiles. The system can use both the supervised model prediction and the cluster profile to give better cost-risk advice.
